# Load and explore FABSA data

Notes for agent design:
- most reviews have multiple aspect labels (avg 1.8, 53% have 2+ labels, max 8)
- sentiment imbalance (65% positive, 31% negative, 4% neutral)
- short reviews (avg 21 words)
- high variation in sentiment across industries (ride hailing 78% negative, consulting 100% neutral, streaming 91% neutral)
- 3 data sources (trustpilot, google play, apple store) with different volumes and industries
- 7 parent and 12 child aspects
- small sample size (<150) for IT, consulting and streaming 

In [1]:
from datasets import load_dataset
import pandas as pd
import ast

pd.set_option('display.max_colwidth', None) 
pd.set_option('display.width', None)   
pd.set_option('display.max_columns', None) 

c:\Users\User\miniconda3\envs\cx-agent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load data from HF and view schema
ds = load_dataset("jordiclive/FABSA")
print(ds)
print(ds["train"].features)
print(f"\nTrain size: {len(ds['train'])}")
print(f"Test size:  {len(ds['test'])}")

DatasetDict({
    train: Dataset({
        features: ['id', 'org_index', 'data_source', 'industry', 'text', 'labels', 'label_codes'],
        num_rows: 7930
    })
    validation: Dataset({
        features: ['id', 'org_index', 'data_source', 'industry', 'text', 'labels', 'label_codes'],
        num_rows: 1057
    })
    test: Dataset({
        features: ['id', 'org_index', 'data_source', 'industry', 'text', 'labels', 'label_codes'],
        num_rows: 1587
    })
})
{'id': Value('int64'), 'org_index': Value('int64'), 'data_source': Value('string'), 'industry': Value('string'), 'text': Value('string'), 'labels': List(List(Value('string'))), 'label_codes': Value('string')}

Train size: 7930
Test size:  1587


In [3]:
# convert to df and preview
df = ds["train"].to_pandas()
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(10)

Shape: (7930, 7)
Columns: ['id', 'org_index', 'data_source', 'industry', 'text', 'labels', 'label_codes']


,id,org_index,data_source,industry,text,labels,label_codes
0,301972057,600,Trustpilot,Price Comparison,"My experience is only around the Parking forum, so my review is based on this specific experience. As someone who needed information on pursuing actions around an unfair parking fine it was pretty good to read, although there are plenty of other very good forums and sites out there too. It’s free so that’s another bonus. But as a normal person, just there looking for information, some discussion as needed and of course a little moral support, I was really shocked by the OTT attitudes and behaviour of some of the regular forum members. Snooty, supercilious, self serving and anti anyone new who’s just finding their way with a specific issue - without the new people who seek info and advice from it, this forum wouldn’t exist would it so why abuse us? I think the administrators need to ‘police’ it better. Just like any forum you come across, it has people there who for them being a regular on it is their life and it comes across that their egos are far more important than the purpose of the forum, which is supposed to be to provide altruistic help to members of the public. I’m afraid I did experience bullying behaviour from ‘regulars’ with repeated attempts to belittle me. There were also some very good people and some good tips but as usual the bad are louder, distract from the reason you’re there and tend to be what you remember.","[[Staff support: Attitude of staff, negative], [Company brand: Reviews, negative], [Company brand: General satisfaction, negative]]","['staff-support.attitude-of-staff.-1', 'company-brand.reviews.-1', 'company-brand.general-satisfaction.-1']"
1,301982453,514,Google Play,Banking,"I love it so handy, plus I hate my bank so it takes away alot of business from them","[[Company brand: General satisfaction, positive], [Company brand: Competitor, positive]]","['company-brand.general-satisfaction.1', 'company-brand.competitor.1']"
2,301980653,369,Google Play,Ride Hailing,Sometimes it takes,"[[Company brand: General satisfaction, negative]]",['company-brand.general-satisfaction.-1']
3,301979991,727,Apple Store,Fashion,This is the worst app I ordered my sneakers 2/1/2 weeks ago still haven’t received them they saying they attempted to send it but never send it thi app should be token down,"[[Logistics rides: Speed, negative], [Online experience: App website, negative], [Company brand: Competitor, negative]]","['logistics-rides.speed.-1', 'online-experience.app-website.-1', 'company-brand.competitor.-1']"
4,301984330,549,Google Play,Travel Booking,So easy & loads of info !,"[[Company brand: General satisfaction, positive]]",['company-brand.general-satisfaction.1']
5,301979193,616,Apple Store,Fashion,"Not all reviews are showing. When it says: this article has 42 reviews, only 2 or sometimes none are showing. Clothes are categorized messy which results in missing items when filtering.","[[Company brand: Reviews, negative], [Online experience: App website, negative]]","['company-brand.reviews.-1', 'online-experience.app-website.-1']"
6,301972213,600,Trustpilot,Price Comparison,very easy to use site and got best deal,"[[Purchase booking experience: Ease of use, positive], [Online experience: App website, positive], [Value: Price value for money, positive]]","['purchase-booking-experience.ease-of-use.1', 'online-experience.app-website.1', 'value.price-value-for-money.1']"
7,301984266,549,Google Play,Travel Booking,"Very useful, especially when you want to try new locations to you and want to understand if it's worth it. Also great as an outlet point when something really is wrong","[[Company brand: General satisfaction, positive]]",['company-brand.general-satisfaction.1']
8,301988812,727,Google Play,Fashion,"Great service, easy returns, friendly people.","[[Staff support: Attitude of staff, positive], [Purchase booking experience: Ease of use, positive], [Company brand: General satisfaction, positive]]","['staff-s

In [4]:
df.dtypes

id              int64
org_index       int64
data_source       str
industry          str
text              str
labels         object
label_codes       str
dtype: object

In [5]:
df.labels.head().tolist()
# array of [aspect, sentiment] pairs

[array([array(['Staff support: Attitude of staff', 'negative'], dtype=object),
        array(['Company brand: Reviews', 'negative'], dtype=object),
        array(['Company brand: General satisfaction', 'negative'], dtype=object)],
       dtype=object),
 array([array(['Company brand: General satisfaction', 'positive'], dtype=object),
        array(['Company brand: Competitor', 'positive'], dtype=object)],
       dtype=object),
 array([array(['Company brand: General satisfaction', 'negative'], dtype=object)],
       dtype=object),
 array([array(['Logistics rides: Speed', 'negative'], dtype=object),
        array(['Online experience: App website', 'negative'], dtype=object),
        array(['Company brand: Competitor', 'negative'], dtype=object)],
       dtype=object),
 array([array(['Company brand: General satisfaction', 'positive'], dtype=object)],
       dtype=object)]

In [6]:
df.label_codes.head().tolist()
# list of 'parent_aspect.child_aspect.sentiment' strings

["['staff-support.attitude-of-staff.-1', 'company-brand.reviews.-1', 'company-brand.general-satisfaction.-1']",
 "['company-brand.general-satisfaction.1', 'company-brand.competitor.1']",
 "['company-brand.general-satisfaction.-1']",
 "['logistics-rides.speed.-1', 'online-experience.app-website.-1', 'company-brand.competitor.-1']",
 "['company-brand.general-satisfaction.1']"]

In [7]:
# parse label_codes into aspects and sentiments
df["label_codes_parsed"] = df["label_codes"].apply(ast.literal_eval)
df_exploded = df.explode("label_codes_parsed")
df_exploded = df_exploded.reset_index(drop=True) 

df_exploded["parent_aspect"] = df_exploded["label_codes_parsed"].apply(lambda x: x.split(".")[0])
df_exploded["child_aspect"] = df_exploded["label_codes_parsed"].apply(lambda x: x.split(".")[1])
df_exploded["sentiment"] = df_exploded["label_codes_parsed"].apply(lambda x: x.split(".")[2])

sentiment_map = {"-1": "negative", "0": "neutral", "1": "positive"}
df_exploded["sentiment"] = df_exploded["sentiment"].map(sentiment_map)

In [8]:
# aspect and sentiment distributions
print(f"Total labels: {len(df_exploded)}\n")
print("=== Parent aspects ===")
print(df_exploded["parent_aspect"].value_counts())
print("\n=== Child aspects ===")
print(df_exploded["child_aspect"].value_counts())
print("\n=== Sentiment ===")
print(df_exploded["sentiment"].value_counts())

Total labels: 13998

=== Parent aspects ===
parent_aspect
company-brand                  3698
online-experience              3654
purchase-booking-experience    2372
value                          1417
staff-support                  1381
logistics-rides                1003
account-management              473
Name: count, dtype: int64

=== Child aspects ===
child_aspect
app-website              3654
general-satisfaction     2905
ease-of-use              2372
attitude-of-staff        1080
price-value-for-money    1015
speed                    1003
competitor                616
account-access            473
discounts-promotions      402
phone                     181
reviews                   177
email                     120
Name: count, dtype: int64

=== Sentiment ===
sentiment
positive    9122
negative    4366
neutral      510
Name: count, dtype: int64


In [9]:
pd.crosstab(df_exploded["child_aspect"], df_exploded["sentiment"]).sort_values("negative", ascending=False)

sentiment,negative,neutral,positive
child_aspect,,,
app-website,1358,277,2019
general-satisfaction,614,15,2276
ease-of-use,555,5,1812
attitude-of-staff,445,9,626
speed,284,2,717
account-access,283,156,34
price-value-for-money,253,3,759
competitor,158,6,452
discounts-promotions,144,30,228


In [10]:
# industry distribution
print(f"Unique industries: {df['industry'].nunique()}\n")
print(df["industry"].value_counts())

Unique industries: 10

industry
Fashion                   2161
Price Comparison          1157
Groceries                 1021
Trading                   1021
Travel Booking             973
Banking                    913
Ride Hailing               383
Information Technology     141
Consulting                  81
Streaming                   79
Name: count, dtype: int64


In [11]:
# data source distribution
print(df["data_source"].value_counts())

data_source
Google Play    4612
Apple Store    1870
Trustpilot     1448
Name: count, dtype: int64


In [12]:
# labels per review
df["num_labels"] = df["label_codes_parsed"].apply(len)
print(df["num_labels"].describe())
print("\nDistribution:")
print(df["num_labels"].value_counts().sort_index())

count    7930.000000
mean        1.765195
std         0.893270
min         1.000000
25%         1.000000
50%         2.000000
75%         2.000000
max         8.000000
Name: num_labels, dtype: float64

Distribution:
num_labels
1    3743
2    2772
3    1032
4     316
5      55
6       9
7       2
8       1
Name: count, dtype: int64


In [13]:
# review length distribution
df["text_length"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().apply(len)

print(df[["text_length", "word_count"]].describe().round(0))

       text_length  word_count
count       7930.0      7930.0
mean         113.0        21.0
std          158.0        29.0
min            3.0         1.0
25%           31.0         6.0
50%           63.0        12.0
75%          131.0        24.0
max         2771.0       488.0


In [14]:
# % sentiment by industry
pd.crosstab(
    df_exploded["industry"],
    df_exploded["sentiment"],
    normalize="index"
).round(2).sort_values("negative", ascending=False)

sentiment,negative,neutral,positive
industry,,,
Ride Hailing,0.78,0.03,0.19
Groceries,0.42,0.01,0.57
Travel Booking,0.32,0.01,0.68
Banking,0.30,0.08,0.62
Trading,0.30,0.01,0.69
Fashion,0.28,0.00,0.72
Price Comparison,0.22,0.00,0.77
Streaming,0.09,0.91,0.00
Information Technology,0.01,0.99,0.00


In [15]:
# sample reviews
for sent in ["positive", "negative", "neutral"]:
    sample = df_exploded[df_exploded["sentiment"] == sent].iloc[0]
    print(f"\n{'='*60}")
    print(f"Sentiment: {sent}")
    print(f"Aspect: {sample['parent_aspect']} -> {sample['child_aspect']}")
    print(f"Industry: {sample['industry']}")
    print(f"Text: {sample['text'][:300]}")


Sentiment: positive
Aspect: company-brand -> general-satisfaction
Industry: Banking
Text: I love it so handy, plus I hate my bank so it takes away alot of business from them

Sentiment: negative
Aspect: staff-support -> attitude-of-staff
Industry: Price Comparison
Text: My experience is only around the Parking forum, so my review is based on this specific experience. As someone who needed information on pursuing actions around an unfair parking fine it was pretty good to read, although there are plenty of other very good forums and sites out there too. It’s free so

Sentiment: neutral
Aspect: online-experience -> app-website
Industry: Banking
Text: How do I check my app and card password if I forgot?


# Dataset summary

In [19]:
# summary stats
print(f"Total reviews:        {len(df)}")
print(f"Total labels:         {len(df_exploded)}")
print(f"Industries:           {df['industry'].nunique()}")
print(f"Parent aspects:       {df_exploded['parent_aspect'].nunique()}")
print(f"Child aspects:        {df_exploded['child_aspect'].nunique()}")
print(f"Avg labels/review:    {df['num_labels'].mean():.1f}")
print(f"Avg words/review:     {df['word_count'].mean():.0f}")
print(f"Sentiment split:      {df_exploded['sentiment'].value_counts(normalize=True).round(2).to_dict()}")

Total reviews:        7930
Total labels:         13998
Industries:           10
Parent aspects:       7
Child aspects:        12
Avg labels/review:    1.8
Avg words/review:     21
Sentiment split:      {'positive': 0.65, 'negative': 0.31, 'neutral': 0.04}


# Test LLM

In [17]:
import requests

response = requests.get("http://localhost:11434/api/tags")
models = response.json().get("models", [])

print(f"Ollama status: OK")
print(f"Models available: {len(models)}")
for m in models:
    print(f"  - {m['name']}")

Ollama status: OK
Models available: 1
  - qwen2.5:3b-instruct


In [18]:
from litellm import completion

response = completion(
    model="ollama/qwen2.5:3b-instruct",
    messages=[{"role": "user", "content": "Reply with exactly: pong"}],
    api_base="http://localhost:11434",
)

answer = response.choices[0].message.content
print(f"Model says: {answer}")

Model says: pong


# End of notebook